# GLDv2 Shard Access Benchmark: S3 Direct vs Google Drive Copy

Measures wall-clock throughput for two strategies to get TAR shards into the Colab `/content` filesystem:

| Strategy | Description |
|----------|-------------|
| **S3 Direct** | `wget` from `s3.amazonaws.com/google-landmark/train/` |
| **Drive Copy** | `cp` / `shutil` from mounted Google Drive archive dir |

Results are reported as **MB/s** (higher is better) and **seconds per shard**.

> **Run guide**: mount Drive (cell 2), set `DRIVE_ARCHIVE_DIR` if needed (cell 3), then run all cells.
> Drive Copy results are skipped automatically if the shard is not present on Drive.

In [ ]:
# Install deps (safe to skip if already installed this session)
!pip install -q tqdm
print('Ready.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# ============================================================
# CONFIGURATION -- edit here
# ============================================================

# S3 base URL for GLDv2 train shards
S3_URL_BASE = 'https://s3.amazonaws.com/google-landmark/train/'

# Where shard TARs live on Drive (set to None to skip Drive tests)
DRIVE_ARCHIVE_DIR = '/content/drive/MyDrive/Vision_Project_2026/GLDv2/archives'

# Local destination for downloaded/copied shards during the benchmark
LOCAL_BENCH_DIR = '/content/bench_shards'
os.makedirs(LOCAL_BENCH_DIR, exist_ok=True)

# Which shard indices to benchmark (keep small -- each shard is ~800 MB-1 GB)
# Use 2-3 shards for reliable averages without wasting too much time.
BENCHMARK_SHARDS = [0, 1, 2]

# Number of repetitions per shard per method (1 is fine; increase for variance)
REPS = 1

# Delete the local copy after each benchmark run (recommended: True)
CLEANUP_AFTER_EACH = True

print('Config loaded.')
print(f'  S3 base       : {S3_URL_BASE}')
print(f'  Drive dir     : {DRIVE_ARCHIVE_DIR}')
print(f'  Local bench   : {LOCAL_BENCH_DIR}')
print(f'  Shard indices : {BENCHMARK_SHARDS}')

In [ ]:
import os
import shutil
import subprocess
import time


def shard_name(idx):
    return f'images_{idx:03d}.tar'


def drive_path(idx):
    """Return Drive path for a shard, or None if not present."""
    if not DRIVE_ARCHIVE_DIR:
        return None
    # Scraper V2 stores shards as train_images_NNN.tar
    for fmt in [f'train_images_{idx:03d}.tar', f'images_{idx:03d}.tar']:
        p = os.path.join(DRIVE_ARCHIVE_DIR, fmt)
        if os.path.exists(p):
            return p
    return None


def file_size_mb(path):
    """Return file size in MB."""
    return os.path.getsize(path) / (1024 * 1024)


def cleanup(path):
    if CLEANUP_AFTER_EACH and os.path.exists(path):
        os.remove(path)
        print(f'  Cleaned up: {os.path.basename(path)}')


def benchmark_s3(idx, rep=1):
    """
    Download shard from S3 using wget, return (size_mb, elapsed_s, mb_per_s).
    Returns None on failure.
    """
    url  = f'{S3_URL_BASE}{shard_name(idx)}'
    dest = os.path.join(LOCAL_BENCH_DIR, f's3_{idx:03d}_rep{rep}.tar')

    # wget: quiet, no resume (fresh download), output to dest
    cmd = ['wget', '-q', '--no-continue', '-O', dest, url]
    print(f'  [S3]   Downloading shard {idx:03d} (rep {rep}) ...')

    t0 = time.perf_counter()
    result = subprocess.run(cmd, capture_output=True)
    elapsed = time.perf_counter() - t0

    if result.returncode != 0 or not os.path.exists(dest) or os.path.getsize(dest) == 0:
        print(f'  [S3]   FAILED (returncode={result.returncode})')
        if os.path.exists(dest):
            os.remove(dest)
        return None

    mb    = file_size_mb(dest)
    speed = mb / elapsed if elapsed > 0 else 0
    print(f'  [S3]   Done: {mb:.1f} MB in {elapsed:.1f}s ({speed:.1f} MB/s)')
    cleanup(dest)
    return mb, elapsed, speed


def benchmark_drive(idx, rep=1):
    """
    Copy shard from Drive to local storage, return (size_mb, elapsed_s, mb_per_s).
    Returns None if shard not present on Drive.
    """
    src = drive_path(idx)
    if src is None:
        print(f'  [Drive] Shard {idx:03d} not found on Drive -- skipping.')
        return None

    dest = os.path.join(LOCAL_BENCH_DIR, f'drive_{idx:03d}_rep{rep}.tar')
    mb   = file_size_mb(src)
    print(f'  [Drive] Copying shard {idx:03d} ({mb:.1f} MB, rep {rep}) ...')

    t0 = time.perf_counter()
    shutil.copy2(src, dest)
    elapsed = time.perf_counter() - t0

    speed = mb / elapsed if elapsed > 0 else 0
    print(f'  [Drive] Done: {mb:.1f} MB in {elapsed:.1f}s ({speed:.1f} MB/s)')
    cleanup(dest)
    return mb, elapsed, speed


print('Helper functions defined.')

## Run Benchmarks

S3 and Drive tests run sequentially for each shard so network and I/O are not contended.

In [ ]:
results = []   # list of dicts

for idx in BENCHMARK_SHARDS:
    print(f'\n{"="*60}')
    print(f'Shard {idx:03d}')
    print(f'{"="*60}')

    for rep in range(1, REPS + 1):
        # --- S3 ---
        r = benchmark_s3(idx, rep)
        if r:
            mb, elapsed, speed = r
            results.append({'shard': idx, 'rep': rep, 'method': 'S3',
                            'size_mb': mb, 'elapsed_s': elapsed, 'speed_mbs': speed})

        # --- Drive ---
        r = benchmark_drive(idx, rep)
        if r:
            mb, elapsed, speed = r
            results.append({'shard': idx, 'rep': rep, 'method': 'Drive',
                            'size_mb': mb, 'elapsed_s': elapsed, 'speed_mbs': speed})

print('\nBenchmark runs complete.')

## Results

In [ ]:
import pandas as pd

if not results:
    print('No results collected. Check that shards are accessible.')
else:
    df = pd.DataFrame(results)

    print('\n=== Per-run results ===')
    print(df[['shard', 'rep', 'method', 'size_mb', 'elapsed_s', 'speed_mbs']]
          .to_string(index=False, float_format=lambda x: f'{x:.2f}'))

    print('\n=== Summary by method ===')
    summary = (
        df.groupby('method')
          .agg(
              runs         =('speed_mbs', 'count'),
              mean_speed   =('speed_mbs', 'mean'),
              median_speed =('speed_mbs', 'median'),
              min_speed    =('speed_mbs', 'min'),
              max_speed    =('speed_mbs', 'max'),
              mean_elapsed =('elapsed_s', 'mean'),
              mean_size_mb =('size_mb',   'mean'),
          )
          .round(2)
    )
    print(summary.to_string())

    if 'S3' in summary.index and 'Drive' in summary.index:
        s3_spd    = summary.loc['S3',    'mean_speed']
        drive_spd = summary.loc['Drive', 'mean_speed']
        ratio     = drive_spd / s3_spd if s3_spd > 0 else float('inf')
        faster    = 'Drive' if drive_spd > s3_spd else 'S3'
        print(f'\n>>> {faster} is faster  ({ratio:.2f}x) <<<')
        print(f'    S3 avg    : {s3_spd:.1f} MB/s')
        print(f'    Drive avg : {drive_spd:.1f} MB/s')
    elif 'S3' in summary.index:
        print('\n(Drive results unavailable -- shards not found on Drive)')
    elif 'Drive' in summary.index:
        print('\n(S3 results unavailable -- check network / shard URL)')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

if results:
    df = pd.DataFrame(results)
    methods = df['method'].unique()

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle('Shard Access Benchmark: S3 vs Google Drive', fontweight='bold')

    # -- Throughput bar chart --
    ax = axes[0]
    colors = {'S3': 'steelblue', 'Drive': 'tomato'}
    for method in methods:
        sub = df[df['method'] == method]
        ax.bar(method, sub['speed_mbs'].mean(), color=colors.get(method, 'grey'),
               alpha=0.85, label=method)
        # error bars if multiple reps
        if len(sub) > 1:
            ax.errorbar(method, sub['speed_mbs'].mean(), yerr=sub['speed_mbs'].std(),
                        fmt='none', color='black', capsize=5)
    ax.set_ylabel('Throughput (MB/s)')
    ax.set_title('Mean Throughput')
    ax.grid(axis='y', alpha=0.3)

    # -- Per-shard throughput line chart --
    ax2 = axes[1]
    for method in methods:
        sub = df[df['method'] == method].groupby('shard')['speed_mbs'].mean()
        ax2.plot(sub.index, sub.values, marker='o', label=method,
                 color=colors.get(method, 'grey'))
    ax2.set_xlabel('Shard index')
    ax2.set_ylabel('Throughput (MB/s)')
    ax2.set_title('Throughput per Shard')
    ax2.legend()
    ax2.grid(alpha=0.3)
    ax2.set_xticks(BENCHMARK_SHARDS)

    plt.tight_layout()
    plt.savefig('/content/shard_benchmark.png', dpi=120, bbox_inches='tight')
    plt.show()
    print('Chart saved to /content/shard_benchmark.png')
else:
    print('No data to plot.')

## Interpreting Results

| Typical Colab speed | Source |
|---------------------|--------|
| 50–150 MB/s | S3 (Google Cloud egress, Colab datacenter proximity) |
| 20–60 MB/s | Google Drive → local (FUSE overhead) |

**Heuristics for the scraper:**

- If **S3 is faster**: download shards fresh every session and skip Drive caching for speed (set `KEEP_TAR_ARCHIVES = False` in the scraper notebook).
- If **Drive is faster**: pre-download shards once to Drive and re-use them across sessions (current default, `KEEP_TAR_ARCHIVES = True`).
- If speeds are similar: Drive wins on *total time* because no re-download cost across sessions.